<a href="https://colab.research.google.com/github/Mark12481632/Reinforcement-Learning/blob/main/Policy_Value_Function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import math

from pprint import pprint

### Policy Iteration

1. Start with an arbitrary policy \(\pi_0\).

2. **Policy evaluation:** calculate \(v_{\pi_k}\) by repeatedly applying

$$
v_{j+1}(s)
=
\sum_a \pi_k(a\mid s)
\sum_{s',r} p(s',r\mid s,a)
\left[r+\gamma v_j(s')\right].
$$

3. **Policy improvement:**

$$
\pi_{k+1}(s)
=
\arg\max_a
\sum_{s',r} p(s',r\mid s,a)
\left[r+\gamma v_{\pi_k}(s')\right].
$$

4. Repeat until the policy no longer changes. The resulting policy is optimal.

In [15]:
# Functions needed for the policy functionality.
# The policy is simply a list of tuples of the form: (action_probability, action, current_state).
# Each entry represents the probability of taking the given action from the current-state.
#
# Clearly the sum of the probability across all possible actions from a given state must be 1.0.
#

def create_random_policy(grid_size = 5):
  """
  Create a default policy - i.e. equal chances of Up=0/Down=1/Left=2/Right=3
  """
  policy = []

  for state in range(grid_size * grid_size):
    for action in [0, 1, 2, 3]:
      policy.append((0.25, action, state))

  return policy


def check_valid_policy(policy):
  """
  Check that the transition probs for all actions from a given state equals 1
  """
  all_states = set(map(lambda x: x[2], policy))
  for state in all_states:
    total_prob = sum(x[0] for x in filter(lambda x: x[2] == state, policy))
    if (total_prob != 1.0):
      raise ValueError(f"Invalid policy file.  State:{state} - Total Prob:{total_prob}")


def policy_actions_from_state(policy, state):
  """
  Return's a list of tuples of the form (action_probability, action) available to
  the policy from a given state.
  """
  matches = filter(lambda x: x[2] == state, policy)
  return list(map(lambda x : (x[0], x[1]), matches))


def improve_policy(env_dynamics, state_function, policy, gamma=1.0):
  """
  Using the given state function and the transition dynamics,
  improve the policy.
  """
  updated_policy = []

  all_states = set(map(lambda x: x[2], policy))
  for state in all_states:
    max_action = 0
    max_value = -math.inf
    all_actions = policy_actions_from_state(policy, state)
    for action_prob, action in all_actions:
      pass


In [12]:
# Functions related to the environment's transition dynamics.

def create_environment_dynamics(grid_size = 5):
  """
  Creates the environment's transition dynamics.
  Holds all the p(prob, next_state, reward, initial_state, action) entries in a list
  Each such entry corresponds to the probablility ("prob") go going to state "next_state" with a
  reward of "reward" when an action "action" is taken from state "initial_state".
  """
  transition_list = []

  for i in range(grid_size):
    for j in range(grid_size):
      for action in range(4):
        if (i == 0 and j == 0) or (i == grid_size - 1 and j == grid_size - 1):
          transition_list.append((1.0, i * grid_size + j, 0, i * grid_size + j, action))
        else:
          if action == 0 and i > 0:
            transition_list.append((1.0, (i - 1) * grid_size + j, -1, i * grid_size + j, action))  # Up
          elif action == 1 and i < grid_size - 1:
            transition_list.append((1.0, (i + 1) * grid_size + j, -1, i * grid_size + j, action))  # Down
          elif action == 2 and j < grid_size - 1:
            transition_list.append((1.0, i * grid_size + (j + 1), -1, i * grid_size + j, action))  # Right
          elif action == 3 and j > 0:
            transition_list.append((1.0, i * grid_size + (j - 1), -1, i * grid_size + j, action))  # Left
          else:
            transition_list.append((1.0, i * grid_size + j, -1, i * grid_size + j, action))

  transition_list = sorted(transition_list, key=lambda x: (x[3], x[4]))

  # print(f"Transition List - entries {len(transition_list)}:")
  # pprint(transition_list)

  return transition_list


def valid_environment(env_dynamics):
  """
  Check that for any initial_state & action the total probability for all next states
  and rewards is 1.0
  """
  all_states = set(map(lambda x: x[3], env_dynamics))
  all_actions = set(map(lambda x: x[4], env_dynamics))
  for state in all_states:
    for action in all_actions:
      total_prob = sum(x[0] for x in filter(lambda x: x[3] == state and x[4] == action,  env_dynamics))
      if (total_prob != 1.0):
        raise ValueError(f"Invalid environment dynamics - state:{state} - total_prob:{total_prob}")


def next_states_from_environment_dynamics(transition_dynamics, state, action):
  """
  Return all (probability, next_state, reward) tuples available from the environment
  when the "action" is taken from the given "state".
  """
  matches = filter(lambda x: x[3] == state and x[4] == action, transition_dynamics)
  return list(map(lambda x : (x[0], x[1], x[2]), matches))


In [13]:
# Functionality relating to the Value Function.

def default_value_function(policy):
  """
  Create a default Value Function - all zeros:
  """
  state_values = []

  all_states = set(map(lambda x: x[2], policy))
  for state in all_states:
    state_values.append((0.0, state))

  return state_values


def improve_value_function(policy, transition_dynamics, initial_state_function, gamma=1.0):
  updated_state_function = []

  all_states = set(map(lambda x: x[2], policy))
  for state_initial in all_states:
    new_state_value = 0.0
    all_actions = policy_actions_from_state(policy, state_initial)
    for action_prob, action in all_actions:
      next_steps = next_states_from_environment_dynamics(transition_dynamics, state_initial, action)
      for prob, next_state, reward in next_steps:
        new_state_value +=  action_prob * prob * (reward + gamma * initial_state_function[next_state][0])

    updated_state_function.append((new_state_value, state_initial))

  return updated_state_function


def value_function_difference(svf1, svf2):
  """
  """
  diff = 0.0
  for v1, v2 in zip(svf1, svf2):
    diff += abs(v1[0] - v2[0])
  return diff

In [16]:
# Sanity Check

GRID_SIZE = 6
GAMMA = 1.0

env_dynamics = create_environment_dynamics(GRID_SIZE)
valid_environment(env_dynamics)

print("-------1")

random_policy = create_random_policy(GRID_SIZE)
check_valid_policy(random_policy)

print("-------2")

state_ftn_old = default_value_function(random_policy)

print("-------3")

delta = 1.0
loop_cnt = 0
while delta > 0.001:
  state_ftn_new = improve_value_function(random_policy, env_dynamics, state_ftn_old, GAMMA)
  delta = value_function_difference(state_ftn_new, state_ftn_old)
  if loop_cnt % 25 == 0:
    print(f"Delta: {delta} at iteration:{loop_cnt}")
  loop_cnt += 1
  state_ftn_old = state_ftn_new
print(f"Delta: {delta} at iteration:{loop_cnt}")

pprint(state_ftn_old)
improve_policy(env_dynamics, state_ftn_old, random_policy, GAMMA)

-------1
-------2
-------3
Delta: 34.0 at iteration:0
Delta: 21.381700087449865 at iteration:25
Delta: 13.930355450913751 at iteration:50
Delta: 9.076893059645641 at iteration:75
Delta: 5.914427840883388 at iteration:100
Delta: 3.853791899819079 at iteration:125
Delta: 2.511098691038679 at iteration:150
Delta: 1.636210983892589 at iteration:175
Delta: 1.0661414437295136 at iteration:200
Delta: 0.694688881340646 at iteration:225
Delta: 0.4526534867363381 at iteration:250
Delta: 0.29494524031991176 at iteration:275
Delta: 0.1921838610248514 at iteration:300
Delta: 0.12522540251328707 at iteration:325
Delta: 0.08159582886397487 at iteration:350
Delta: 0.053167162208161756 at iteration:375
Delta: 0.03464327989098592 at iteration:400
Delta: 0.022573272519210263 at iteration:425
Delta: 0.01470855628664225 at iteration:450
Delta: 0.009583972720434986 at iteration:475
Delta: 0.006244836768345863 at iteration:500
Delta: 0.00406908360444902 at iteration:525
Delta: 0.0026513809718835546 at iterat

NameError: name 'state_action_possibilites' is not defined